# Notebook for causal analysis of the SHAPly results using DoWhy-library

DoWhy

1 Predictor -> {multiple confounder}


**Features, that occur in all 3 models**

**3x**
* Health condition
* I generally feel that what I do in life is worthwhile
* Can't find the way because life has become so complicated?
* I feel I am free to decide how to live my life
* I am optimistic about the future
* Household able to make ends meet?
* Personal financial situation
* How much trust the police?


**2x**
* Deprivation index: No. of items hhold can't afford (RF, XG)
* Quality of education system? (RF, XG)
* The value of what I do is not recognised by others? (RF, XG)
* Marital status (SVR, XG)
* Can most people be trusted? (SVR, XG)

**1x**

**Only RF:**
Can afford to pay for a week's annual holiday away?
Quality of health services?
Feel left out of society?

**Only SVR:**
Financial situation of your household compared to 12 months ago?
Education completed
Quality of long term care services?

**Only XGBoost:**
I seldom have time to do the things I really enjoy
Neighbourhood problems – traffic

In [2]:
from dowhy import CausalModel
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd

### Feature list with corresponding confounders

In [3]:
outcome = "How happy are you?"

In [4]:
feature_confounder_map = {
    "Health condition": [
        "Age",
        "Income quartiles",
        "Chronic health problems?",
        "Education completed",
        "Employment - 7 groups"
    ],

    "I generally feel that what I do in life is worthwhile": [
        "A person to get support from when feeling depressed",
        "How frequently participate in social activities?",
        "Employment - 7 groups",
        "Marital status",
        "Health condition"
    ],

    "Can't find the way because life has become so complicated?": [
        "Education completed",
        "Employment - 7 groups",
        "A person to get support from when feeling depressed",
        "Age",
        "Household size"
    ],

    "I feel I am free to decide how to live my life": [
        "Income quartiles",
        "Education completed",
        "How much trust the government?"
    ],

    "I am optimistic about the future": [
        "Personal financial situation",
        "Access to recreational or green areas?",
        "A person to get support from to raise emergency money",
        "Health condition",
        "Age",
        "Employment - 7 groups"
    ],

    "Household able to make ends meet?": [
        "Employment - 7 groups",
        "Income quartiles",
        "Personal financial situation",
        "Household size",
        "No. of children"
    ],

    "Personal financial situation": [
        "Employment - 7 groups",
        "Can afford a meal with meat/chicken/fish every second day?",
        "Household structure",
        "Income quartiles",
        "Education completed"
    ],

    "How much trust the police?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the government?",
        "How much trust the legal system?",
        "Rural/urban living",
        "Age"
    ],

    "Deprivation index: No. of items hhold can't afford": [
        "Income quartiles",
        "Employment - 7 groups",
        "Can afford to keep home adequately warm?",
        "Household size",
        "Household structure"
    ],

    "Quality of education system?": [
        "Education completed",
        "How much trust the legal system?",
        "Income quartiles",
        "Rural/urban living",
        "Age"
    ],

    "The value of what I do is not recognised by others?": [
        "Employment - 7 groups",
        "How frequently participate in social activities?",
        "A person to get support from when feeling depressed",
        "Education completed",
        "Marital status"
    ],

    "Marital status": [
        "Age",
        "Household structure",
        "How frequently participate in social activities?",
        "No. of children",
        "Education completed"
    ],

    "Can most people be trusted?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the press?",
        "How much trust the government?",
        "Education completed",
        "Rural/urban living"
    ]
}

### Run causal analysis using doWhy

In [5]:
# Data
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')

# Reversing the scale on some features (higher values = more positive)
data["Health condition"] = 6 - data["Health condition"]
data["Can't find the way because life has become so complicated?"] = 6 - data["Can't find the way because life has become so complicated?"]
data["Household able to make ends meet?"] = 7 - data["Household able to make ends meet?"]
data["A person to get support from when feeling depressed"] = 3 - data["A person to get support from when feeling depressed"]
data["I feel I am free to decide how to live my life"] = 6 - data["I feel I am free to decide how to live my life"]
data["I generally feel that what I do in life is worthwhile"] = 6 - data["I generally feel that what I do in life is worthwhile"]
data["I am optimistic about the future"] = 6 - data["I am optimistic about the future"]
data["Marital status"] = (data["Marital status"] == 1).astype(int)

In [6]:
def calculate_causal_values(confounder_map):
    # Save results
    results = []
    # Iterate through all features
    for treatment, confounders in confounder_map.items():
        try:
            print(f"Treatment: {treatment}")
            model = CausalModel(
                data=data,
                treatment=treatment,
                outcome=outcome,
                common_causes=confounders
            )

            identified_estimand = model.identify_effect()

            estimate = model.estimate_effect(
                identified_estimand,
                method_name="backdoor.linear_regression"
            )

            results.append({
                "Treatment": treatment,
                "ATE (DML)": estimate.value,
                "Confounders": ", ".join(confounders)
            })
        except Exception as e:
            results.append({
                "Treatment": treatment,
                "ATE (DML)": None,
                "Confounders": ", ".join(confounders),
                "Error": str(e)
            })
    return results




# Results
results_df = pd.DataFrame(calculate_causal_values(feature_confounder_map))
results_df.sort_values(by="ATE (DML)", ascending=False, inplace=True, key=abs)
display(results_df)

Treatment: Health condition
Treatment: I generally feel that what I do in life is worthwhile
Treatment: Can't find the way because life has become so complicated?
Treatment: I feel I am free to decide how to live my life
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?
Treatment: Deprivation index: No. of items hhold can't afford
Treatment: Quality of education system?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

Treatment: The value of what I do is not recognised by others?
Treatment: Marital status
Treatment: Can most people be trusted?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]


,Treatment,ATE (DML),Confounders
0,Health condition,0.655323,"Age, Income quartiles, Chronic health problems..."
1,I generally feel that what I do in life is wor...,0.605259,A person to get support from when feeling depr...
11,Marital status,0.521154,"Age, Household structure, How frequently parti..."
3,I feel I am free to decide how to live my life,0.499720,"Income quartiles, Education completed, How muc..."
6,Personal financial situation,0.491358,"Employment - 7 groups, Can afford a meal with ..."
2,Can't find the way because life has become so ...,-0.442613,"Education completed, Employment - 7 groups, A ..."
10,The value of what I do is not recognised by ot...,0.394884,"Employment - 7 groups, How frequently particip..."
4,I am optimistic about the future,0.388772,"Personal financial situation, Access to recrea..."
5,Household able to make ends meet?,0.341952,"Employment - 7 groups, Income quartiles, Perso..."
8,Deprivation index: No. of items hhold can't af...,-0.325283,"Income quartiles, Employment - 7 groups, Can a..."


In [7]:
results_df.to_csv(path_or_buf="Results/results_causal_analysis.csv")

# Refutation Tests

Refutation tests are implemented to check whether results stay stable.

In [8]:
from dowhy import CausalModel


def run_refutation_tests(confounder_map, data, outcome):
    refutation_results = []

    for treatment, confounders in confounder_map.items():
        print(f"\nTreatment: {treatment}")

        try:
            model = CausalModel(
                data=data,
                treatment=treatment,
                outcome=outcome,
                common_causes=confounders
            )

            identified_estimand = model.identify_effect()

            estimate = model.estimate_effect(
                identified_estimand,
                method_name="backdoor.linear_regression"
            )

            original_ate = estimate.value

            refuters = {
                "Placebo Treatment": {
                    "method_name": "placebo_treatment_refuter",
                    "placebo_type": "permute",
                    "num_simulations": 100,
                    "random_seed": 42
                },
                "Random Common Cause": {
                    "method_name": "random_common_cause",
                    "num_simulations": 100,
                    "random_seed": 42
                },
                "Data Subset": {
                    "method_name": "data_subset_refuter",
                    "subset_fraction": 0.8,
                    "num_simulations": 100,
                    "random_seed": 42
                }
            }

            for test_name, refuter_args in refuters.items():
                try:
                    refuter = model.refute_estimate(
                        identified_estimand,
                        estimate,
                        **refuter_args
                    )

                    refutation_results.append({
                        "Treatment": treatment,
                        "Original ATE": original_ate,
                        "Refutation Test": test_name,
                        "Refuted Estimate": refuter.new_effect,
                        "Difference": refuter.new_effect - original_ate,
                        "Absolute Difference": abs(refuter.new_effect - original_ate),
                        "Result": str(refuter)
                    })

                except Exception as e:
                    refutation_results.append({
                        "Treatment": treatment,
                        "Original ATE": original_ate,
                        "Refutation Test": test_name,
                        "Refuted Estimate": None,
                        "Difference": None,
                        "Absolute Difference": None,
                        "Result": f"Error: {e}"
                    })

        except Exception as e:
            refutation_results.append({
                "Treatment": treatment,
                "Original ATE": None,
                "Refutation Test": "Model failed",
                "Refuted Estimate": None,
                "Difference": None,
                "Absolute Difference": None,
                "Result": f"Error: {e}"
            })

    return pd.DataFrame(refutation_results)

In [9]:
# Run test
refutation_df = run_refutation_tests(
    confounder_map=feature_confounder_map,
    data=data,
    outcome=outcome
)

display(refutation_df)

C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers


Treatment: Health condition


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers


Treatment: I generally feel that what I do in life is worthwhile


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

...


Treatment: Marital status


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers


Treatment: Can most people be trusted?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

,Treatment,Original ATE,Refutation Test,Refuted Estimate,Difference,Absolute Difference,Result
0,Health condition,0.655323,Placebo Treatment,0.004068,-6.512547e-01,6.512547e-01,Refute: Use a Placebo Treatment\nEstimated eff...
1,Health condition,0.655323,Random Common Cause,0.655312,-1.055309e-05,1.055309e-05,Refute: Add a random common cause\nEstimated e...
2,Health condition,0.655323,Data Subset,0.655629,3.064594e-04,3.064594e-04,Refute: Use a subset of data\nEstimated effect...
3,I generally feel that what I do in life is wor...,0.605259,Placebo Treatment,-0.002031,-6.072900e-01,6.072900e-01,Refute: Use a Placebo Treatment\nEstimated eff...
4,I generally feel that what I do in life is wor...,0.605259,Random Common Cause,0.605281,2.219138e-05,2.219138e-05,Refute: Add a random common cause\nEstimated e...
5,I generally feel that what I do in life is wor...,0.605259,Data Subset,0.603420,-1.838723e-03,1.838723e-03,Refute: Use a subset of data\nEstimated effect...
6,Can't find the way because life has become so ...,-0.442613,Placebo Treatment,-0.002131,4.404822e-01,4.404822e-01,Refute: Use a Placebo Treatment\nEstimated eff...
7,Can't find the way because life has become so ...,-0.442613,Random Common Cause,-0.442624,-1.104474e-05,1.104474e-05,Refute: Add a random common cause\nEstimated e...
8,Can't find the way because life has become so ...,-0.442613,Data Subset,-0.441519,1.094549e-03,1.094549e-03,Refute: Use a subset of data\nEstimated effect...
9,I feel I am free to decide how to live my life,0.499720,Placebo Treatment,-0.000893,-5.006129e-01,5.006129e-01,Refute: Use a Placebo Treatment\nEstimated eff...


In [10]:
refutation_summary = refutation_df[
    [
        "Treatment",
        "Original ATE",
        "Refutation Test",
        "Refuted Estimate",
        "Difference",
        "Absolute Difference"
    ]
].copy()

display(refutation_summary)

refutation_summary.to_csv(
    "Results/refutation_tests_summary.csv",
    index=False
)

,Treatment,Original ATE,Refutation Test,Refuted Estimate,Difference,Absolute Difference
0,Health condition,0.655323,Placebo Treatment,0.004068,-6.512547e-01,6.512547e-01
1,Health condition,0.655323,Random Common Cause,0.655312,-1.055309e-05,1.055309e-05
2,Health condition,0.655323,Data Subset,0.655629,3.064594e-04,3.064594e-04
3,I generally feel that what I do in life is wor...,0.605259,Placebo Treatment,-0.002031,-6.072900e-01,6.072900e-01
4,I generally feel that what I do in life is wor...,0.605259,Random Common Cause,0.605281,2.219138e-05,2.219138e-05
5,I generally feel that what I do in life is wor...,0.605259,Data Subset,0.603420,-1.838723e-03,1.838723e-03
6,Can't find the way because life has become so ...,-0.442613,Placebo Treatment,-0.002131,4.404822e-01,4.404822e-01
7,Can't find the way because life has become so ...,-0.442613,Random Common Cause,-0.442624,-1.104474e-05,1.104474e-05
8,Can't find the way because life has become so ...,-0.442613,Data Subset,-0.441519,1.094549e-03,1.094549e-03
9,I feel I am free to decide how to live my life,0.499720,Placebo Treatment,-0.000893,-5.006129e-01,5.006129e-01
